In [1]:
from scipy.signal import butter, filtfilt, find_peaks
import neurokit2 as nk
import numpy as np
import pyxdf
import matplotlib.pyplot as plt
import pandas as pd
from biosppy.signals.ppg import ppg
import heartpy as hp
import glob
import re
import os

debugging = False
def Trace(message):
    """
    Function to print messages with a specific format.
    """
    if (debugging):
        print(f"[Trace] {message}")
    else:
        pass


In [21]:
def get_gaze_data(filepath = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-13/sub-13_ses-13_task-Baseline/_.xdf"):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    gaze_stream = next((s for s in streams if s["info"]["name"][0] == "GazePointStream"), None)
    if gaze_stream:
        gaze_data = gaze_stream['time_series']
        gaze_ts = gaze_stream['time_stamps']
        return gaze_data, gaze_ts
    else:
        print(f"Warning: Gaze stream not found in {filepath}.")
        return None, None

gaze_data, gaze_ts = get_gaze_data()
# convert to DataFrame with time stamps in one axis and the object being looked at in the other axis
gaze_df = pd.DataFrame({'time': gaze_ts, 'layer': [l[0] for l in gaze_data]})
# print each unique layer
print(gaze_df["layer"].unique())

['0']


In [ ]:
baseline_processed = []
task_processed = []

def filter_bvp(signal, lowcut=0.5, highcut=8.0, fs=400):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high <= low:
        print(f"Warning: Highcut frequency ({highcut}Hz) is not above lowcut frequency ({lowcut}Hz) at fs={fs}Hz. Adjusting filter or skipping.")
        return signal
    try:
        b, a = butter(4, [low, high], btype='bandpass')
        filtered = filtfilt(b, a, signal)
        # plot before and after filtering for debugging
        if debugging:
            plt.figure(figsize=(12, 6))
            plt.subplot(2, 1, 1)
            plt.plot(signal, label='Original Signal')
            plt.title('Original Signal')
            plt.subplot(2, 1, 2)
            plt.plot(filtered, label='Filtered Signal', color='orange')
            plt.title('Filtered Signal')
            plt.tight_layout()
            plt.show()
        return filtered
    except ValueError as ve:
        print(f"ValueError during filtering: {ve}. Returning unfiltered signal.")
        return signal

def get_hrv_metrics(bvp_segment, timestamps, fs=400):
    if len(bvp_segment) < fs * 10:
        print(f"Warning: Segment too short ({len(bvp_segment)/fs:.2f}s, need at least 10s), skipping HRV.")
        return pd.Series(dtype=float)
    try:
        filtered_segment = filter_bvp(bvp_segment, fs=fs)
        wd, m = hp.process(filtered_segment, sample_rate=fs, calc_freq=False, high_precision=True, clean_rr=True)
        peaks = wd.get('peaklist', [])
        if len(peaks) < 5:
            print(f"Warning: Not enough peaks found ({len(peaks)}, need at least 5), skipping HRV.")
            return pd.Series(dtype=float)
        # Ensure peak indices are integers and within bounds before indexing timestamps
        valid_peaks = []
        for p in peaks:
            try:
                p_int = int(p) # Convert to integer
                if 0 <= p_int < len(timestamps): # Check bounds
                    valid_peaks.append(p_int)
            except (ValueError, TypeError):
                print(f"Warning: Invalid peak value {p} encountered, skipping it.")

        if len(valid_peaks) < 2:
            print(f"Warning: Not enough valid peaks ({len(valid_peaks)}) after filtering and conversion. Original peaks count from heartpy: {len(peaks)}. Skipping HRV.")
            return pd.Series(dtype=float)
        peak_times_sec = timestamps[valid_peaks]
        ibi_ms = np.diff(peak_times_sec) * 1000
        if len(ibi_ms) < 3:
            print(f"Warning: Not enough IBIs calculated ({len(ibi_ms)}), skipping HRV.")
            return pd.Series(dtype=float)
        ibi_event_times_sec = peak_times_sec[1:]
        hrv_indices = nk.hrv({'RRI': ibi_ms, 'RRI_Time': ibi_event_times_sec}, sampling_rate=1000)
        return hrv_indices.iloc[0]
    except Exception as e:
        print(f"Error processing segment: {e}")
        return pd.Series(dtype=float)

def extract_bvp_and_markers_from_xdf(filepath):
    #print(f"Attempting to load XDF: {filepath}")
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None, pd.DataFrame(), {'bvp': False, 'markers': False}
    bvp_s = next((s for s in streams if s["info"]["name"][0] == "OpenSignals" and any(ch['label'][0] == 'BVP0' for ch in s['info']['desc'][0]['channels'][0]['channel'])), None)
    mark_s = next((s for s in streams if s["info"]["name"][0] == "UnityMarkers"), None)
    streams_found = {'bvp': False, 'markers': False}
    bvp_signal, bvp_ts = None, None
    df_events = pd.DataFrame()
    if bvp_s:
        bvp_data_raw = bvp_s['time_series']
        bvp_ts_raw = bvp_s['time_stamps']
        bvp_channel_idx = next((i for i, ch in enumerate(bvp_s['info']['desc'][0]['channels'][0]['channel']) if ch['label'][0] == 'BVP0'), None)
        if bvp_channel_idx is not None and bvp_data_raw.ndim == 2 and bvp_data_raw.shape[1] > bvp_channel_idx:
            bvp_signal = bvp_data_raw[:, bvp_channel_idx].astype(np.float64)
            bvp_ts = bvp_ts_raw
            streams_found['bvp'] = True
            Trace(f"BVP stream found and BVP0 channel extracted from {filepath}.")
        else:
            print(f"Warning: BVP0 channel not found or data format unexpected in OpenSignals stream for {filepath}.")
    else:
        print(f"Warning: BVP stream (OpenSignals with BVP0) not found in {filepath}.")
    if mark_s:
        markers_raw = mark_s['time_series']
        marker_ts_raw = mark_s['time_stamps']
        if len(markers_raw) > 0:
            df_events = pd.DataFrame({'time': marker_ts_raw, 'event': [m[0] for m in markers_raw]})
            streams_found['markers'] = True
            #print(f"UnityMarkers stream found in {filepath}.")
        #else:
            #print(f"Warning: UnityMarkers stream found but no event data in {filepath}.")
    return bvp_signal, bvp_ts, df_events, streams_found

def get_gaze_data(filepath):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    gaze_stream = next((s for s in streams if s["info"]["name"][0] == "GazePointStream"), None)
    if gaze_stream:
        gaze_data = gaze_stream['time_series']
        gaze_ts = gaze_stream['time_stamps']
        return gaze_data, gaze_ts
    else:
        print(f"Warning: Gaze stream not found in {filepath}.")
        return None, None
    
def process_subject_data(subject_id, baseline_filepath, task_filepath, nominal_srate=400):
    results_list = []
    condition_from_task_folder = "Unknown"  # Default

    # --- Determine Condition from Task File's PARENT FOLDER (if it exists) ---
    if task_filepath:
        task_parent_folder_name = os.path.basename(os.path.dirname(task_filepath))
        match_cond = re.search(r'_task-([^_\s]+)', task_parent_folder_name) # More robust regex for condition
        if match_cond:
            parsed_condition = match_cond.group(1)
            # Ensure 'Baseline' from folder name doesn't become the experimental condition
            if parsed_condition.lower() != 'baseline':
                condition_from_task_folder = parsed_condition
            else:
                print(f"Warning: Task filepath {task_filepath} seems to be from a folder named '...task-Baseline...'. Experimental condition remains 'Unknown' or will be based on a non-baseline task folder if available.")
        else:
            print(f"Warning: Could not parse condition from task folder name: {task_parent_folder_name} for subject {subject_id}")

    # --- 1. Process Baseline File ---
    if baseline_filepath:
        Trace(f"Processing BASELINE for subject {subject_id} (Experimental Condition: {condition_from_task_folder}) from: {os.path.basename(baseline_filepath)}")
        bvp_baseline, ts_baseline, df_events_baseline, streams_baseline = extract_bvp_and_markers_from_xdf(baseline_filepath)

        if streams_baseline['bvp'] and bvp_baseline is not None and ts_baseline is not None:
            Trace(f"Calculating baseline HRV for {subject_id} (full file)...")
            hrv_baseline_metrics = get_hrv_metrics(bvp_baseline, ts_baseline, fs=nominal_srate)
            if not hrv_baseline_metrics.empty:
                baseline_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Baseline'}
                baseline_res.update(hrv_baseline_metrics)
                results_list.append(baseline_res)
                print(f"Baseline HRV calculated for {subject_id}.")
                baseline_processed.append(subject_id)
            else:
                print(f"No HRV metrics obtained for baseline for subject {subject_id}.")
        else:
            print(f"Could not process BVP for baseline for subject {subject_id} from {os.path.basename(baseline_filepath)}.")
    else:
        print(f"No baseline filepath provided for subject {subject_id}.")

    # --- 2. Process Task File ---
    if task_filepath:
        Trace(f"Processing TASK for subject {subject_id}, Condition: {condition_from_task_folder} from: {os.path.basename(task_filepath)}")
        bvp_task_full, ts_task_full, df_events_task, streams_task = extract_bvp_and_markers_from_xdf(task_filepath)

        if streams_task['bvp'] and bvp_task_full is not None and ts_task_full is not None:
            Trace(f"Calculating task HRV for {subject_id} (full file {len(bvp_task_full)/nominal_srate:.2f}s)...")
            hrv_task_metrics = get_hrv_metrics(bvp_task_full, ts_task_full, fs=nominal_srate)
            if not hrv_task_metrics.empty:
                task_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Task'}
                task_res.update(hrv_task_metrics)
                results_list.append(task_res)
                print(f"Task HRV calculated for {subject_id}.")
                task_processed.append(subject_id)
            else:
                print(f"No HRV metrics obtained for task phase for subject {subject_id}.")
        else:
            print(f"Could not process BVP for task for subject {subject_id} from {os.path.basename(task_filepath)}.")
    else:
        print(f"No task filepath provided for subject {subject_id}.")

    return pd.DataFrame(results_list) if results_list else pd.DataFrame()

# --- Main Processing Loop ---
all_participant_dfs = [] 
data_root_folder = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data"
nominal_srate_main = 400

session_folders = [f.path for f in os.scandir(data_root_folder) if f.is_dir() and re.match(r'ses-\d+', f.name)]
Trace(f"Found session folders: {session_folders}")

for session_folder_path in session_folders:
    session_name = os.path.basename(session_folder_path)
    Trace(f"\nProcessing session: {session_name}")

    potential_st_folders = [f.path for f in os.scandir(session_folder_path) if f.is_dir() and "_task-" in f.name and "sub-" in f.name]

    subject_folders_map = {}
    for st_folder_path in potential_st_folders:
        st_folder_name = os.path.basename(st_folder_path)
        match_sub = re.search(r'(sub-[^_\s]+)', st_folder_name)
        if match_sub:
            subject_id_key = match_sub.group(1)
            if subject_id_key not in subject_folders_map:
                subject_folders_map[subject_id_key] = []
            subject_folders_map[subject_id_key].append(st_folder_path)
        else:
            print(f"  Warning: Could not parse subject ID from folder name {st_folder_name} in session {session_name}.")

    Trace(f"  Found data for subjects in {session_name}: {list(subject_folders_map.keys())}")

    for subject_id_key, folders_for_subject in subject_folders_map.items():
        Trace(f"    Processing data for subject key: {subject_id_key} in session: {session_name}")

        baseline_filepath = None
        task_condition_filepath = None

        for folder_path in folders_for_subject:
            folder_name = os.path.basename(folder_path)
            xdf_files_in_folder = glob.glob(os.path.join(folder_path, "*.xdf"))

            if not xdf_files_in_folder:
                print(f"      Warning: No XDF file found in folder {folder_name}. Skipping this folder.")
                continue
            if len(xdf_files_in_folder) > 1:
                print(f"      Warning: Multiple XDF files found in {folder_name}. Using the first one: {os.path.basename(xdf_files_in_folder[0])}.")
            current_xdf_file = xdf_files_in_folder[0]

            if "_task-Baseline" in folder_name:
                if baseline_filepath:
                    print(f"      Warning: Multiple baseline folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                baseline_filepath = current_xdf_file
                Trace(f"      Found Baseline file: {os.path.basename(baseline_filepath)} in folder {folder_name}")
            elif "_task-" in folder_name:
                if task_condition_filepath:
                    print(f"      Warning: Multiple task condition folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                task_condition_filepath = current_xdf_file
                Trace(f"      Found Task Condition file: {os.path.basename(task_condition_filepath)} in folder {folder_name}")

        if baseline_filepath and task_condition_filepath:
            cleaned_subject_id = re.sub(r'^sub-', '', subject_id_key)
            Trace(f"      Pair found for {cleaned_subject_id}: Baseline (baseline_filepath), Task (task_condition_filepath)")
            #print(f"      Pair found for {cleaned_subject_id}: Baseline ({os.path.basename(baseline_filepath)}), Task ({os.path.basename(task_condition_filepath)})")
            subject_hrv_df = process_subject_data(
                subject_id=cleaned_subject_id,
                baseline_filepath=baseline_filepath,
                task_filepath=task_condition_filepath,
                nominal_srate=nominal_srate_main
            )
            if not subject_hrv_df.empty:
                all_participant_dfs.append(subject_hrv_df)
            else:
                print(f"      No HRV data generated for subject {cleaned_subject_id} in session {session_name}.")
        else:
            missing_parts = []
            if not baseline_filepath: missing_parts.append("baseline file")
            if not task_condition_filepath: missing_parts.append("task condition file")
            print(f"      Skipping subject {subject_id_key} in session {session_name} due to missing { ' and '.join(missing_parts) }.")

if not all_participant_dfs:
    print("\nNo dataframes to concatenate. Final DataFrame will be empty.")
    final_df = pd.DataFrame()
else:
    final_df = pd.concat(all_participant_dfs, ignore_index=True)
    print("\n--- Combined Results DataFrame ---")
    if not final_df.empty:
        print(final_df.head())
        print(f"Total number of subjects processed: {len(final_df)}")
        print(f"Baseline processed subjects: {len(baseline_processed)}")
        print(f"Task processed subjects: {len(task_processed)}")
    else:
        print("Final DataFrame is empty after processing all subjects.")

Baseline HRV calculated for 10.
Task HRV calculated for 10.
Could not process BVP for baseline for subject 11 from _.xdf.
Could not process BVP for task for subject 11 from _.xdf.
      No HRV data generated for subject 11 in session ses-11.
Could not process BVP for baseline for subject 12 from _.xdf.
Could not process BVP for task for subject 12 from _.xdf.
      No HRV data generated for subject 12 in session ses-12.
Could not process BVP for baseline for subject 13 from _.xdf.
Could not process BVP for task for subject 13 from _.xdf.
      No HRV data generated for subject 13 in session ses-13.
Baseline HRV calculated for 6.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 6.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 7.
Task HRV calculated for 7.


...

Baseline HRV calculated for 2.
Task HRV calculated for 2.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Baseline HRV calculated for 3.
Task HRV calculated for 3.
Baseline HRV calculated for 4.
Task HRV calculated for 4.
Error processing segment: 
----------------
Could not determine best fit for given signal. Please check the source signal.
 Probable causes:
- detected heart rate falls outside of bpmmin<->bpmmax constraints
- no detectable heart rate present in signal
- very noisy signal (consider filtering and scaling)
If you're sure the signal contains heartrate data, consider filtering and/or scaling first.
----------------

No HRV metrics obtained for baseline for subject 5.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning: Duplicate x values detected. Averaging their corresponding y values.
  warn(


Task HRV calculated for 5.

--- Combined Results DataFrame ---
  ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0            10  MediumFi  Baseline  660.764801   135.314503   31.175987   
1            10  MediumFi      Task  661.646606   815.397120  163.615276   
2             6  MediumFi  Baseline  720.362129   343.066159   74.287398   
3             6  MediumFi      Task  708.597926  1232.896508  363.475236   
4             7     LowFi  Baseline  778.079340  1238.519766  167.767156   

   HRV_SDNNI1  HRV_SDANN2  HRV_SDNNI2  HRV_SDANN5  ...  HRV_SampEn  \
0  123.251099   17.586581  124.454787         NaN  ...    0.748622   
1  525.514743   87.186293  544.865326         NaN  ...    0.260511   
2  322.452551         NaN         NaN         NaN  ...    0.946400   
3  909.627909  173.201379  944.014767         NaN  ...    0.324930   
4  803.354401  110.833837  847.796769         NaN  ...    0.161545   

   HRV_ShanEn  HRV_FuzzyEn  HRV_MSEn  HRV_CMSEn  HRV_RCMSEn

In [3]:
print(final_df)

   ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0             10  MediumFi  Baseline  660.764801   135.314503   31.175987   
1             10  MediumFi      Task  661.646606   815.397120  163.615276   
2              6  MediumFi  Baseline  720.362129   343.066159   74.287398   
3              6  MediumFi      Task  708.597926  1232.896508  363.475236   
4              7     LowFi  Baseline  778.079340  1238.519766  167.767156   
5              7     LowFi      Task  683.793288  1287.271150  234.905801   
6              8    HighFi  Baseline  758.057742  1390.552859  642.005356   
7              8    HighFi      Task  833.953731  2765.923004  333.962601   
8              9  MediumFi  Baseline  656.686601   127.146566   19.098847   
9              9  MediumFi      Task  659.704301   100.910997   10.671450   
10             1    HighFi  Baseline  715.679284   226.257083   48.481917   
11             1    HighFi      Task  727.953191  1189.775585  273.490521   

# Statistical Modeling\n
\n
We now have a DataFrame (`final_df`) containing HRV metrics for each participant, condition, and phase. We can use this to perform a mixed-design analysis.\n
\n
A Linear Mixed-Effects Model (LMM) is suitable here. It allows us to model:\n
- **Fixed Effects:** The average effects of `Phase` (Baseline vs. Task) and `Condition` (LowFi, MedFi, HighFi), and their interaction (`Phase * Condition`).\n
- **Random Effects:** The variability between participants. We assume each participant has their own baseline level of the HRV metric, modeled as a random intercept (`1|ParticipantID`).\n
\n
We will model one HRV metric at a time, for example, RMSSD (Root Mean Square of Successive Differences).

In [4]:
import statsmodels.formula.api as smf

# Check if final_df exists and has data
if 'final_df' in locals() and not final_df.empty and 'HRV_RMSSD' in final_df.columns:
    # Ensure necessary columns are not all NaN
    if final_df[['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID']].isnull().all().any():
        print("Warning: One or more critical columns contain only NaN values. Cannot run model.")
    else: 
        # Remove rows with NaN in the outcome variable or predictors
        model_df = final_df.dropna(subset=['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID'])
        
        if model_df.empty:
            print("Warning: No valid data remaining after removing NaNs. Cannot run model.")
        else:
            #print("\n--- Inspecting model_df before fitting LMM ---")
            #print(model_df.info())

            # Print a few rows to see actual values
            print("\n--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---")
            # Ensure ParticipantID, Phase, and Condition are appropriate types
            model_df['ParticipantID'] = model_df['ParticipantID'].astype('category')
            model_df['Phase'] = model_df['Phase'].astype('category')
            model_df['Condition'] = model_df['Condition'].astype('category')
            
            # Define the model formula for fixed effects
            fixed_effects_formula = "HRV_RMSSD ~ C(Phase) * C(Condition)"
            # Define the random effects formula (random intercept for ParticipantID)
            random_effects_formula = "~1"
            
            try:
                # Fit the model using re_formula for random effects
                model = smf.mixedlm(fixed_effects_formula, 
                                  model_df, 
                                  groups=model_df["ParticipantID"], 
                                  re_formula=random_effects_formula)
                result = model.fit()
                
                # Print the summary
                print(result.summary())
            except Exception as e:
                print(f"Error fitting model: {e}")
                print("\nPlease check data structure and variability.")
                print("Model DataFrame head:")
                print(model_df.head())
else:
    print("Skipping statistical analysis: 'final_df' not created or is empty or missing 'HRV_RMSSD' column.")


--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---
                            Mixed Linear Model Regression Results
Model:                        MixedLM             Dependent Variable:             HRV_RMSSD  
No. Observations:             19                  Method:                         REML       
No. Groups:                   10                  Scale:                          263614.5408
Min. group size:              1                   Log-Likelihood:                 -108.4369  
Max. group size:              2                   Converged:                      Yes        
Mean group size:              1.9                                                            
---------------------------------------------------------------------------------------------
                                            Coef.    Std.Err.   z    P>|z|   [0.025   0.975] 
---------------------------------------------------------------------------------------------
Intercept                     

# Interpretation (Example for RMSSD)\n
\n
Look at the model summary table:\n
- **Intercept:** Estimated RMSSD for the reference group (e.g., Baseline phase in the LowFi condition, depending on how statsmodels encodes categories).\n
- **C(Phase)[T.Task]:** The estimated average *change* in RMSSD when moving from Baseline to Task phase (holding Condition constant at the reference level).\n
- **C(Condition)[T.MedFi/HighFi]:** The estimated average *difference* in RMSSD between MedFi/HighFi and the reference condition (LowFi) during the reference phase (Baseline).\n
- **C(Phase)[T.Task]:C(Condition)[T.MedFi/HighFi]:** The interaction effect. This shows how the *effect of Phase* (Task vs. Baseline) differs between the MedFi/HighFi conditions compared to the LowFi condition. A significant interaction suggests the task effect depends on the fidelity level.\n
- **P>|z|:** The p-value for each coefficient. Values < 0.05 typically indicate statistical significance.\n
- **Group Var:** The variance of the random intercepts, indicating how much baseline RMSSD varies between participants.\n
\n
*Note: You would repeat the modeling step for other HRV metrics of interest (e.g., SDNN, MeanNN, pNN50) by changing the dependent variable in the formula.*

In [ ]:
import pingouin as pg
pg.mea
emms = pg.emmeans(model, dv='HRV_RMSSD',
                  between='Condition', within='Phase')
pg.plot_emmeans(emms)


AttributeError: module 'pingouin' has no attribute 'emmeans'